In [ ]:
import os
from ftplib import FTP
import time
import socket

In [ ]:
FTP_HOST = 'ftp.ptree.jaxa.jp'
FTP_USER = 'shuzu6918_gmail.com'   
FTP_PASS = 'SP+wari8'      

# 保存路径
SAVE_DIR = '/proj6/users/ma/data/Himawari8_LST'

# 年份 
YEARS = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

# 月份 
MONTHS = [6, 7, 8]

In [ ]:
def connect_ftp():
    """连接FTP并返回对象"""
    try:
        ftp = FTP(FTP_HOST)
        ftp.login(FTP_USER, FTP_PASS)
        ftp.set_pasv(True) # 被动模式，防止防火墙卡住
        return ftp
    except Exception as e:
        print(f"FTP连接失败: {e}")
        return None

def download_main():
    if not os.path.exists(SAVE_DIR):
        os.makedirs(SAVE_DIR)

    ftp = connect_ftp()
    if not ftp: return

    print("开始下载")

    for year in YEARS:
        for month in MONTHS:
            for day in range(1, 32):
                date_str = f"{year}-{month:02d}-{day:02d}"
                
                # JAXA目录: /jma/netcdf/YYYYMM/DD/
                remote_dir = f"/jma/netcdf/{year}{month:02d}/{day:02d}/"
                
                try:
                    ftp.cwd(remote_dir)
                except:
                    # 如果目录不存在(比如6月31日)，跳过
                    continue

                print(f"正在扫描: {date_str} ...")
                
                try:
                    file_list = ftp.nlst()
                except (socket.error, Exception):
                    print("连接超时，尝试重连...")
                    ftp.close()
                    ftp = connect_ftp()
                    ftp.cwd(remote_dir)
                    file_list = ftp.nlst()

                # 筛选文件
                for filename in file_list:
                    
                    # .nc 文件
                    if not filename.endswith(".nc"): continue

                    # 2. 5km 分辨率 (02401)
                    # 06001 (2km)
                    if "02401_02401" not in filename: continue

                    # 3. 必须是整点 (00-23)
                    # JAXA文件名: ..._HHMM_...
                    is_hourly = False
                    for h in range(24):
                        if f"_{h:02d}00_" in filename:
                            is_hourly = True
                            break
                    if not is_hourly: continue
                        
                    # 构造本地保存路径 (按月归档)
                    local_month_dir = os.path.join(SAVE_DIR, f"{year}{month:02d}")
                    if not os.path.exists(local_month_dir):
                        os.makedirs(local_month_dir)

                    local_filepath = os.path.join(local_month_dir, filename)

                    # 检查是否已存在 (简单检查大小)
                    if os.path.exists(local_filepath):
                        # 如果文件非空(大于1KB)，则跳过
                        if os.path.getsize(local_filepath) > 1024:
                            continue

                    # 开始下载
                    print(f"下载 -> {filename}")
                    try:
                        with open(local_filepath, 'wb') as f:
                            ftp.retrbinary('RETR ' + filename, f.write)
                    except Exception as e:
                        print(f"下载出错 {filename}: {e}")
                        # 失败删除，防止坏文件
                        if os.path.exists(local_filepath):
                            os.remove(local_filepath)
                        # 重连一下以防万一
                        ftp = connect_ftp()
                        ftp.cwd(remote_dir)

    ftp.quit()
    print("完成")

In [ ]:
if __name__ == "__main__":
    download_main()